In [4]:
import pandas as pd
import numpy as np

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

In [3]:
df = pd.read_csv('data/processed/daily_ridership_weather_2025.csv', parse_dates=['date']).sort_values('date')

weather_cols = ['temperature_mean_c', 'rain_mm', 'snowfall_cm']

features = df[weather_cols].copy()
features[['rain_mm', 'snowfall_cm']] = np.log1p(
    features[['rain_mm', 'snowfall_cm']]
)

X = StandardScaler().fit_transform(features)

In [5]:
labels_by_k = {}
comparison_rows = []

for k in range(2, 6):
    model = KMeans(n_clusters=k, random_state=21, n_init=20)
    labels = model.fit_predict(X)
    labels_by_k[k] = labels

    sizes = pd.Series(labels).value_counts().sort_index()

    comparison_rows.append({
        "k": k,
        "silhouette": silhouette_score(X, labels),
        "smallest_cluster": sizes.min(),
        "cluster_sizes": sizes.to_dict()
    })

comparison = (
    pd.DataFrame(comparison_rows)
    .sort_values("silhouette", ascending=False)
    .reset_index(drop=True)
)

display(comparison.round({"silhouette": 3}))

,k,silhouette,smallest_cluster,cluster_sizes
0,3,0.508,24,"{0: 271, 1: 70, 2: 24}"
1,4,0.495,17,"{0: 129, 1: 65, 2: 17, 3: 154}"
2,5,0.460,17,"{0: 108, 1: 109, 2: 65, 3: 17, 4: 66}"
3,2,0.373,130,"{0: 130, 1: 235}"


In [7]:
chosen_k = int(comparison.iloc[0]["k"])

clustered_days = df.assign(
    cluster=labels_by_k[chosen_k],
    rain_day_pct = 100 * df['rain_mm'].gt(0),
    snow_day_pct = 100 * df['snowfall_cm'].gt(0)
)

profiles = (
    clustered_days.groupby('cluster')
    .agg(
        days = ('date', 'size'),
        median_temp_c = ('temperature_mean_c', 'median'),
        rain_days_pct = ('rain_day_pct', 'mean'),
        snow_days_pct = ('snow_day_pct', 'mean'),
        median_rides = ('total_rides', 'median')
    )
    .sort_values('median_rides', ascending=False)
)

print(f"Selected k = {chosen_k}")
display(profiles.round(1))

Selected k = 3


,days,median_temp_c,rain_days_pct,snow_days_pct,median_rides
cluster,,,,,
1,70,19.0,100.0,2.9,18470.5
0,271,10.5,27.7,8.9,16176.0
2,24,-2.0,50.0,100.0,3567.0


# Clustering Discussion
We chose 3 clusters since it gave the highest silhouette score (which measures consistency between clusters) of 0.508. It gave us three easily interpretable weather profiles. Cluster 1 corresponds with warmer, rainy days. Cluster 0 corresponds with mild, mostly dry days. Cluster 2 corresponds with cold, snowy days. We see that cold, snowy days had the lowest median ridership at 3,567 rides per day, while warmer, rainy days had the highest median ridership at 18,471 rides per day.